# Intermediate Object-Oriented Programming
## 1. Overloading and Multiple Inheritance

## 1.1 Overloading the + operator
- Use the `__add__()` magic method to overload `+`
- `self` and `other` are passed to `__add__()`

In [245]:
# Overload the + operator
class Team:
    def __init__(self, team_members):
        self.team_members = team_members
    
    def __add__(self, other):
        return Team(self.team_members + other.team_members)

In [246]:
# Adding two objects
team1 = Team(["Timm", "Sar"])
team2 = Team(["Bob", "Al"])
dream_team = team1 + team2
dream_team.team_members

['Timm', 'Sar', 'Bob', 'Al']

#### Use + to create a new type of object
What if we want to create a `Team` by combining `Employee` objects?
- `__add__` is implemented in the `Employee` class
- Creates a new object of `Team` by creating a list of `Employee` names
- The result of adding two `Employee` objects is a single `Team`

In [247]:
class Team:
    def __init__(self, members):
        self.members = members
        
class Employee:
    def __init__(self, name, title):
        self.name = name
        self.title = title
    
    def __add__(self, other):
        return Team([self.name, other.name]) # Use + to create a team with the name of each Employee

In [248]:
timm = Employee("Timm", "Data Scientist")
sar = Employee("Sar", "CEO")
# Attempt to add these together to create a team
top_team = timm + sar
top_team.members

['Timm', 'Sar']

### Overloading other operators
Operator | Magic Method | Operator Type

`-`      | `__sub__`    | Arithmetic

!= | `__ne__` | Comparison

< | `__lt__` | Comparison

`>` | `__gt__` | Comparison

+= | `__iadd__` | Assignment

and | `__contains__` | Logical

in | `__contains__` | Membership

is | `__is__` | Identity


### 1. Exercise: Creating a Network of Computers

In [249]:
class Network:
    def __init__(self, ip_addresses):
        self.ip_addresses = ip_addresses

class Computer:
    def __init__(self, operating_system, ip_address):
        self.operating_system = operating_system
        self.ip_address = ip_address
        
    def __add__(self, other):
        if self.operating_system == other.operating_system:
            return Network([self.ip_address, other.ip_address])
        raise Exception("Incompatible operating systems.") 
    
# Build a network using Morgan and Jenny's Computers
morgans_comp = Computer("Windows", "182.112.81.991")
jennys_comp = Computer("Windows", "177.511.64.162")
network = morgans_comp + jennys_comp

In [250]:
# Check the network
network.ip_addresses

['182.112.81.991', '177.511.64.162']

## 1.2 Multiple Inheritance
- Allows for a class to inherit the functionality of more than a single class
- Maintains an 'is-a' relationship

In [251]:
# Creating an Employee and Student class
class Employee:
    def __init__(self, department):
        self.department = department
        
    def begin_job(self):
        print(f"Welcome to {self.department}!")
        
class Student:
    def __init__(self, school):
        self.school = school
        self.courses = []
    
    def add_courses(self, course_name):
        self.courses.append(course_name)
        
# Define an Intern that inherits both classes
class Intern(Employee, Student):
    def __init__(self, department, school, duration):
        # Make a call to BOTH constructors
        Employee.__init__(self, department)
        Student.__init__(self, school)
        self.duration = duration

In [252]:
# Creating an Intern object
stephen = Intern("Software development", "Echo University", 10)
stephen.begin_job() # Method from Employee
stephen.add_courses("Intermediate OOP") # Method from Intern
print(stephen.courses)

Welcome to Software development!
['Intermediate OOP']


### 1.2.1 Multilevel inheritance
- Inherit a class which inherits from another class, becoming a grandchild
- Maintain an 'is-a' relationship
- _Method resolution order (MRO):_ The order in which Python determines which method is used when parent and children implement a method with the same name:
1. Children will be searched first
2. Parent classes will be searched left-to-right as they were defined in class statement
3. `.mro()` method and `__mro__` attribute

In [253]:
# Calling mro
Intern.__mro__

(__main__.Intern, __main__.Employee, __main__.Student, object)

## 2. Customs Class Features and Type Hints

### 2.1 Type Hints
Optional tool that allows for information about an object to be added to the code.
- Easier to read, troubleshoot
- Not enforced by the interpreter
- Built-in keywords, `typing` library, custom classes
- Use the syntax `variable: type`
- `def ...() -> type:` to specify return type

In [254]:
# Quick Introduction:
from typing import List, Dict
# Add type hinting when creating a variable
gpa: float = 3.92
# Add type hinting to a function/ method definition
def check_grades(year:str) -> List[int]:
    pass
# Can be used for Dictionaries
students: Dict[str, float] = {"Casey": 3.71, "Sarah": 4.0}
# Or custom classes
class Student:
    def __init(self, name:str, student_id:int) -> None:
        pass # here the rest
    
# Use Student to type hint
# walker: Student = Student("Sarah Walker", 313131)   

#### Example: Type hinting with custom classes

In [255]:
class Agent:
  def __init__(self, codename: str, missions: int):
    self.codename: str = codename
    self.missions: int = missions

  def add_mission(self, location: str) -> None:
    self.missions += 1
    print(f"{self.codename} completed a mission in " + \
          f"{location}. This was mission #{self.missions}")

# Create an Agent object, add type hints
chuck: Agent = Agent("Charles Carmichael", 37)

# Create a list of locations, add a mission for each
locations: List[str] = ["Burbank", "Paris", "Prague"]
for location in locations:
  chuck.add_mission(location)

Charles Carmichael completed a mission in Burbank. This was mission #38
Charles Carmichael completed a mission in Paris. This was mission #39
Charles Carmichael completed a mission in Prague. This was mission #40


### 2.2 Descriptors
Descriptors are objects used to regulate the way that an attribute is retrieved, set, or deleted.
- Create descriptors with `@property`
- "getter", "setter", "deleter"
- Method name == name of attribute

In [256]:
# Creating descriptors with @property
class Student:
    def __init__(self, name, ssn):
        self.name = name
        self.ssn = ssn # Doesn't need to include an underscore
        
    @property
    def ssn(self): # This is the "getter" method, controls how ssn is retrieved, interact with self._ssn not self.ssn
        return "XXX-XX-" + self._ssn[-4:]
    
    @ssn.setter # validate data quality, interact with self._ssn not self.ssn
    def ssn(self, new_ssn):
        if len(new_ssn) == 11:
            self._ssn = new_ssn
        
    @ssn.deleter
    def ssn(self): # Can perform clean up, soft delete, raise exception
        raise AttributeError("Can't delete SSN")

#### Example 1: Student SSN

In [257]:
# Descriptor in action
shaw = Student("Daniel Shaw", "193-80-1821")
print(shaw.ssn)  # Access the ssn attribute
shaw.ssn = "821-11-9380" # Update Shaw's social security number
print(shaw.ssn)
# Try to delete it
try:
    del shaw.ssn
except:
    print("AttributeError: Can't delete SSN")

XXX-XX-1821
XXX-XX-9380
AttributeError: Can't delete SSN


#### Example 2: Bank Account Email

In [258]:
class BankAccount:
  def __init__(self, email):
    self.email = email
    
  @property
  def email(self):
    return f"Email for this account is: {self._email}"
  
  @email.setter
  def email(self, new_email_address):
    if "@" in new_email_address:
      self._email = new_email_address
    else:
      print("Please make sure to enter a valid email.")
  
  # Define a method to be used when deleting the email attribute
  @email.deleter
  def email(self):
    del self._email
    print("Email deleted, make sure to add a new email!")

In [259]:
# email.property
email1 = BankAccount('timm@mail.com')
print(email1.email)
# email.setter
email1.email = 'timm.webmail.com'
# delete email
del email1.email

Email for this account is: timm@mail.com
Please make sure to enter a valid email.
Email deleted, make sure to add a new email!


### 2.3 Customizing Attribute Access

#### 2.3.1 `__getattr__()`
`__getattr__()` is executed when an attempt to reference **ANY attribute** outside of an object's namespace is made
- magic method, not called directly
- takes a `name` parameter
- implements custom functionality, rather than raising an `AttributeError`

In [260]:
# Resolving AttributeError
class Student:
    def __init__(self, student_name, major):
        self.student_name = student_name
        self.major = major
        
    def __getattr__(self, name):
        print(f"{name} does not exist in the object's namespace, try setting a value for {name} first")

# Create an object
karina = Student('Karina','English')
karina.residence_hall

residence_hall does not exist in the object's namespace, try setting a value for residence_hall first


#### 2.3.2 `__setattr__()`
`__setattr__()` is a magic method that is executed when a (new or existing) attribute is **set** or **updated**
- incl. attributes set using `__init__()`
- takes `name` of attribute and `value`
- leverages `__dict__` attribute of the object
- `__dict__` stores all attributes of the object, can be used to retrieve and store data

In [261]:
# Customizing attribute storage
class Student:
    def __init__(self, student_name, major):
        self.student_name = student_name
        self.major = major
    
    def __setattr__(self, name, value):
        # if value is a string, set the attr using __dict__ attr
        if isinstance(value, str):
            print(f"Setting {name} = {value}")
            self.__dict__[name] = value
            
        else: # Otherwise, raise an exception noting an incorrect data type
            raise Exception("Unexpected data type!")

In [262]:
# In action
karina = Student('Karina','English')
karina.residence_hall = "Honors College South"

Setting student_name = Karina
Setting major = English
Setting residence_hall = Honors College South


In [263]:
# Set an attribute using a value of type 'int'
try:
    karina.student_id = 19301872
except:
    print("Exception: Unexpected data type!")

Exception: Unexpected data type!


#### 2.3.3 Using `__getattr__` and `__setattr__` together


In [264]:
class Student:
    def __init__(self, student_name, major):
        self.student_name = student_name
        self.major = major
        
    def __getattr__(self, name):
        # Set the attr with a placeholder
        self.__setattr__(name, None)
        return None
    
    def __setattr__(self, name, value):
        if value is None: # Print a message denoting a placeholder
            print(f"Setting placeholder for {name}")
        
        self.__dict__[name] = value # Set the attr

In [265]:
# In action - Set the object
karina = Student('Karina','English')

# Check the known attributes
print(karina.student_name)
print(karina.major)

# Ask for an unknown attribute
karina.lastname
karina.lastname = 'Storch' # set it to a value
print(karina.lastname) # Check again

Karina
English
Setting placeholder for lastname
Storch


### 2.4 Custom Iterators
Classes that allow for a collection of objects or data stream to be traversed, and return one item at a time:
- Looped over using `for` loops
- `next()` function

#### Iterator protocol
`__iter__()`
- Returns an iterator, in this case, a reference to itself
- `... return self`

`__next__()`
- Returns the next value in the collection or data stream
- Iteration, transformation, and generation takes place

Both `__iter__()` and `__next__()` must be defined for a class to be considered an iterator!

In [266]:
import random
# Using an example iterator
class CoinFlips:
    def __init__(self, number_of_flips):
        self.number_of_flips = number_of_flips
        self.count = 0
        
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.count < self.number_of_flips:
            self.count += 1
            return random.choice(["H","T"])
        else: # Stop execution with StopIteration
            raise StopIteration

#### StopIteration
- Signals end of collections/ data stream
- Prevents infinite loops

In [267]:
three_flips = CoinFlips(3)
while True:
    try:
        print(next(three_flips))

    except StopIteration:
        print("Completed all coin flips!")
        break

T
T
H
Completed all coin flips!


In [268]:
five_flips = CoinFlips(5)
# But also possible
for i in five_flips:
    print(i)

H
T
H
T
T


## 3. Object-oriented design patterns
### 3.1 Abstract Base Classes
Abstract base classes create a blueprint for classes by defining abstract methods that must be implemented by all children. 
- Cannot be instantiated
- Defines methods that must be implemented by subclasses
- Is usually created with Python's `abc` module
- `@abstractmethod` decorator - also possible to implement multiple methods

In [269]:
from abc import ABC, abstractmethod
class School(ABC):
    @abstractmethod
    def enroll(self):
        pass # This method must be implemented in classes that inherit from it
    
    @abstractmethod
    def add_course(self, course_name):
        pass

    def graduate(self):
        print("Congrats on graduating!") # Concrete methods are inherited

In [270]:
class HighSchool(School):
    def __init__(self):
        self.courses = []
    
    def enroll(self):
        print("Welcome to high school!")
        
    def add_course(self, course_name): # Implementing abstract method
        self.courses.append(course_name)
        print(f"You enrolled in {course_name}")
        
# Create an instance of Highschool
high_school = HighSchool()
high_school.enroll()
high_school.graduate()

Welcome to high school!
Congrats on graduating!


#### Example: Implementing Abstract Base Classes

In [271]:
from abc import ABC, abstractmethod
from typing import Dict

class Company(ABC):
    @abstractmethod
    def create_budget(self):
        pass
    
    def hire_employee(self, name):
        print(f"Welcome to the team, {name}!")
        
class Technology(Company):
    def __init__(self, name):
        self.name = name
        
    def create_budget(self, year, expenses:Dict[str, int]):
        for expense, amount in expenses.items():
            print(f"{year} budget for {expense} is {amount}")

# Create an instance of the Technology class, call methods
t = Technology("Tina's Tech Advisors")
t.create_budget(2024, {"Salaries": 100000, "Supplies": 500})
t.hire_employee("Christian")
            

2024 budget for Salaries is 100000
2024 budget for Supplies is 500
Welcome to the team, Christian!


### 3.2 Interfaces
A special kind of class made up of only abstract methods that create a contract with the classes that implement the interface
- Must define abstract methods
- Skeleton of method defintiions
- Parameters do not need to match
- Formal or Informal

#### Example: Informal Interfaces

In [272]:
class YogurtSupplier:
  def __init__(self):
    self.orders = {}
  
  # Finish defining the take_order() method
  def take_order(self, product_name, quantity):
    self.orders[f"{product_name}_{quantity}"] = {
      "product_name": product_name, "quantity": quantity
    }
  
  # Implement a make_delivery() abstract method
  def make_delivery(self, order_id, location):
    print(f"Delivering order: {order_id} to {location}")
    del self.orders[order_id]

#### Example: Formal interface with ABC

In [273]:
# Create a Product interface
class Product(ABC):
  
  # Define a purchase() abstract method
  @abstractmethod
  def purchase(self, quantity):
    pass
  
  # Create an update_price() abstract method
  @abstractmethod
  def update_price(self, new_price):
    pass

### 3.3 Factory methods
Design pattern that uses factory methods to create objects that are used in another method
- return objects that implement an interface.
- reduce complexity in a method.
- reuseable, modular.
- `_` to denote a factory method

In [274]:
class Customer(ABC):
  @abstractmethod
  def make_payment(self, price):
    pass

class RewardsMember(Customer):
  def make_payment(self, price):
    print(f"""Total price for rewards member is ${price * .90}, which is 10% off.""")

class NewCustomer(Customer):
  def make_payment(self, price):
    print(f"""Total price for new customer is ${price}""")

In [275]:
class Checkout:
  # Create a _get_customer() factory method 
  def _get_customer(self, customer_type):
    if customer_type == "Rewards Member":
      return RewardsMember()
    elif customer_type == "New Customer":
      return NewCustomer()
  
  # Define the complete_transaction() method
  def complete_transaction(self, customer_type, price):
    customer = self._get_customer(customer_type)
    customer.make_payment(price)

In [276]:
checkout = Checkout()

checkout.complete_transaction("Rewards Member", 100)
checkout.complete_transaction("New Customer", 100)

Total price for rewards member is $90.0, which is 10% off.
Total price for new customer is $100
